In [ ]:
import pandas as pd
import numpy as np
import re
from functools import reduce
import seaborn as sns
import missingno as msno
%matplotlib inline

import matplotlib.pyplot as plt
from scipy.stats import norm


In [ ]:
# Set the figure size - handy for larger output
from matplotlib import pyplot as plt
plt.rcParams["figure.figsize"] = [10, 6]
# Set up with a higher resolution screen (useful on Mac)
%config InlineBackend.figure_format = 'retina'

## import data

In [ ]:
data = pd.read_csv('D:\RESEARCH IN AFRICA\DATA_PREPROCESSING/WOS_OLD/is_african_countries_corresp_author_all/studies_with_african_all_based_on_affil_only.csv').drop(['Unnamed: 0.1','Unnamed: 0','Addresses'], axis=1)

data2 = data.rename(columns={"UT (Unique WOS ID)": "wos_ID"})
data2['wos_ID'] = data2['wos_ID'].str.lower()
data2

In [ ]:
data_prep = pd.read_csv('D:\RESEARCH IN AFRICA\Manuscripts/comprarison of databases/data/preprocessing_wos/wos_data_preprocessing_clean.csv')[['wos_ID', 'PubMed_ID_wos', 'doi','source_title','year']]
data_prep = data_prep.drop_duplicates('wos_ID')
data_prep

In [ ]:
data_concat = data2.merge(data_prep, on = 'wos_ID', how='left')
data_concat

### years

In [ ]:
data_concat = data_concat.loc[data_concat['year'] != 2022.0]

In [ ]:
data_concat.loc[data_concat['year']<=1981.0, 'year_group'] = '1975-1981'
data_concat.loc[data_concat['year'].between(1982.0,1986.0), 'year_group'] = '1982-1986'
data_concat.loc[data_concat['year'].between(1987.0,1991.0), 'year_group'] = '1987-1991'
data_concat.loc[data_concat['year'].between(1992.0,1996.0), 'year_group'] = '1992-1996'
data_concat.loc[data_concat['year'].between(1997.0,2001.0), 'year_group'] = '1997-2001'
data_concat.loc[data_concat['year'].between(2002.0,2006.0), 'year_group'] = '2002-2006'
data_concat.loc[data_concat['year'].between(2007.0,2011.0), 'year_group'] = '2007-2011'
data_concat.loc[data_concat['year'].between(2012.0,2016.0), 'year_group'] = '2012-2016'
data_concat.loc[data_concat['year']>2016, 'year_group'] = '2017-2021'

In [ ]:
data_concat.shape

## list of countries

In [ ]:
data.columns.to_list()

In [ ]:
list_african_countries =  ['Algeria', 'Eritrea', 'Ghana', 'Mauritania', 'Mauritius', 'Congo Republic', 'Equatorial Guinea', 'South Sudan', 
 'Uganda', "Cote d'Ivoire", 'Kenya', 'Lesotho', 'Niger', 'Central African Republic', 'Sao Tome and Principe', 'Togo', 
 'Tanzania', 'Mozambique', 'Zambia', 'Guinea-Bissau', 'Benin', 'Guinea', 'Senegal', 'Egypt', 'Cameroon', 'Zimbabwe', 
 'DR Congo', 'Madagascar', 'Nigeria', 'Seychelles', 'Tunisia', 'Liberia', 'Burundi', 'Namibia', 'Rwanda', 'Malawi', 
 'Mali', 'South Africa', 'Eswatini', 'Cabo Verde', 'Somalia', 'Djibouti', 'Libya', 'Morocco', 'Gambia', 'Angola', 'Comoros',
 'Ethiopia', 'Sierra Leone', 'Botswana', 'Chad', 'Sudan', 'Burkina Faso', 'Gabon']

In [ ]:
len(list_african_countries)

In [ ]:
north_africa = ['Egypt','Djibouti', 'Libya', 'Morocco','Tunisia','Algeria']

In [ ]:
eastern_africa = ['Ethiopia','Eritrea','South Sudan', 'Kenya','Uganda', 'Tanzania', 'Rwanda', 'Mauritius',
                  'Seychelles','Madagascar','Sudan','Somalia','Comoros','Burundi']

In [ ]:
Southern_africa = ['South Africa','Mozambique','Zambia','Namibia','Malawi','Zimbabwe','Lesotho','Angola','Botswana','Eswatini']

In [ ]:
Western_africa = ['Ghana','Mauritania',"Cote d'Ivoire",'Niger','Togo','Guinea-Bissau','Benin','Guinea','Senegal', 'Sierra Leone',
                  'Burkina Faso','Liberia','Mali','Cabo Verde','Gambia','Nigeria']

In [ ]:
Central_africa = ['Congo Republic','Equatorial Guinea','Central African Republic','Sao Tome and Principe',
                  'Cameroon','DR Congo','Chad','Gabon']

In [ ]:
non_african_countries = ['United States','Australia','Belgium','Brazil', 'Canada', 'Czech Republic', 'Denmark',
 'United Kingdom', 'Germany','France', 'Hungary','India', 'Israel','Italy', 'Japan','Kuwait','Lebanon','Micronesia, Fed. Sts.',
'Netherlands','New Zealand','Norway','Papua New Guinea','Serbia','Singapore','Sweden','Switzerland',
'Russia','Afghanistan','Austria','Belarus','Bulgaria','Fiji','Finland','China', 'Indonesia', 'Iran', 'Iraq',
 'Ireland', 'Poland', 'Portugal', 'Romania', 'Saudi Arabia', 'Spain','Ukraine', 'Venezuela', 'Greece', 'Trinidad and Tobago',
 'Bangladesh', 'Malaysia', 'Pakistan', 'Syria', 'Taiwan', 'Thailand', 'Paraguay', 'Colombia', 'Cook Islands',
 'Iceland', 'Jamaica', 'Jordan', 'Monaco', 'New Caledonia', 'Soviet Union (former)', 'United Arab Emirates', 'Yemen', 'Bolivia',
 'Chile', 'Peru', 'Mexico', 'Qatar', 'Solomon Islands', 'Uruguay', 'Argentina', 'South Korea', 'Cuba', 'Haiti',
 'Nepal', 'Philippines', 'Belize', 'Cyprus', 'Mongolia', 'Samoa', 'Sri Lanka', 'Brunei Darussalam', 'Dominican Republic',
 'Oman', 'Costa Rica', 'Honduras', 'Malta', 'Ecuador', 'Georgia', 'Myanmar', 'El Salvador', 'Panama', 'Vatican',
 'Marshall Islands', 'Bahrain', 'Guyana', 'Nicaragua', 'Suriname', 'Barbados', 'Guatemala', 'Vietnam', 'Croatia', 'Luxembourg',
 'Vanuatu', 'Slovakia', 'Tonga', 'Latvia', 'Lithuania', 'Slovenia', 'Laos', 'Albania', 'Kyrgyz Republic', 'Uzbekistan', 'Armenia','Estonia',
 'Moldova', 'Tajikistan', 'Azerbaijan', 'St. Lucia', 'Kazakhstan', 'Bosnia and Herzegovina', 'Liechtenstein',
 'Macedonia', 'Grenada', 'Bahamas', 'Cambodia', 'Palestine', 'St. Kitts and Nevis', 'St. Vincent and the Grenadines', 'Maldives', 'Bhutan',
 'Andorra', 'Antigua and Barbuda', 'Dominica', 'Montenegro', 'Bermuda', 'North Korea', 'Kosovo', 'Reunion', 'Anguilla', 'San Marino',
 'Timor-Leste', 'Turkmenistan', 'Turkey','Curacao']

In [ ]:
scimago_asiatic_region = ['Japan','North Korea','India','Laos','Philippines','Taiwan','Thailand','Afghanistan','Pakistan',
                  'Timor-Leste', 'Turkmenistan','Tajikistan','China','Indonesia','Cambodia','Uzbekistan',  'Maldives', 
                  'Bhutan','Myanmar','Kazakhstan','Sri Lanka','Brunei Darussalam','Bangladesh','Malaysia','South Korea',
                  'Singapore','Nepal','Mongolia','Kyrgyz Republic','Vietnam'] 
# Hong Kong
# Macao

In [ ]:
scimago_eastern_europe = ['Russia','Poland','Czech Republic','Romania','Ukraine','Hungary','Slovakia','Croatia','Serbia',
                  'Latvia', 'Lithuania', 'Slovenia',  'Albania',  'Armenia',  'Estonia', 'Moldova','Bosnia and Herzegovina',
                  'Belarus','Bulgaria','Georgia','Azerbaijan','Macedonia','Montenegro']

In [ ]:
scimago_latin_america = ['Brazil','Chile', 'Peru', 'Mexico','Uruguay', 'Argentina',  'Cuba', 'Haiti','Paraguay', 'Colombia','Venezuela', 
                 'Ecuador','Costa Rica','Panama','Trinidad and Tobago','Jamaica','Bolivia','Nicaragua', 'Suriname', 'Barbados', 
                 'Guatemala','Dominican Republic','Honduras','Grenada','El Salvador','St. Kitts and Nevis', 
                 'St. Vincent and the Grenadines', 'Bahamas','Guyana','Belize','Dominica','Antigua and Barbuda','St. Lucia',
                 'Anguilla','Curacao']
# Puerto Rico  # Guadeloupe  # French Guiana  # Martinique   # Falkland Islands (Malvinas)  # Netherlands Antilles # Cayman Islands
# Virgin Islands (U.S.) # Aruba  # Virgin Islands (British) # Montserrat  # Turks and Caicos Islands  # Curaçao 
# South Georgia and the South Sandwich Islands  # Sint Maarten

In [ ]:
scimago_middle_east = ['Iran', 'Iraq','Saudi Arabia','United Arab Emirates', 'Yemen','Qatar','Oman','Kuwait','Lebanon','Jordan','Bahrain',
               'Palestine', 'Israel','Turkey','Syria']
# Egypt

In [ ]:
scimago_northern_america = ['United States','Bermuda','Canada']
# United States Minor Outlying Islands
# Saint Pierre and Miquelon

In [ ]:
scimago_pacific_region = ['Australia','New Zealand','Fiji','Papua New Guinea','New Caledonia','Solomon Islands','Samoa','Tonga','Vanuatu',
                  'Micronesia, Fed. Sts.','Marshall Islands','Cook Islands'] 
# French Polynesia   # Guam # Palau # Kiribati # Tuvalu # Nauru  # Niue  # Norfolk Island # Wallis and Futuna
# Cocos (Keeling) Islands  # Tokelau  # Christmas Island  # French Southern Territories  # Pitcairn 
# Heard Island and McDonald Islands

In [ ]:
scimago_western_europe = ['Belgium','Denmark','United Kingdom', 'Germany','France', 'Italy', 'Netherlands','Norway','Sweden','Switzerland',
                  'Austria','Finland', 'Ireland', 'Portugal', 'Spain', 'Greece','Iceland', 'Monaco','Luxembourg','Malta',
                  'Liechtenstein','Andorra','Cyprus','Vatican','San Marino']
# Greenland   # Faroe Islands  # Gibraltar   # Svalbard and Jan Mayen  # Bouvet Island

In [ ]:
scimago_arab_countries = ['Egypt','Djibouti', 'Libya', 'Morocco','Tunisia','Algeria','Iraq','Saudi Arabia','United Arab Emirates','Yemen',
                  'Qatar','Oman','Kuwait','Lebanon','Jordan','Bahrain','Palestine', 'Syria','Sudan','Mauritania', 'Somalia','Comoros']

In [ ]:
scimago_arab_countries_non_africa = ['Iraq','Saudi Arabia','United Arab Emirates','Yemen',
                  'Qatar','Oman','Kuwait','Lebanon','Jordan','Bahrain','Palestine', 'Syria']

In [ ]:
scimago_arab_countries_africa = ['Egypt','Djibouti', 'Libya', 'Morocco','Tunisia','Algeria',
                  'Sudan','Mauritania', 'Somalia','Comoros']

In [ ]:
scimago_BRIICS = ['China','India','Russia','Brazil','South Africa','Indonesia']

In [ ]:
scimago_BRIICS_not_south_africa = ['China','India','Russia','Brazil','Indonesia']

In [ ]:
scimago_EU_28 = ['Belgium','Denmark','United Kingdom', 'Germany','France', 'Italy', 'Netherlands','Austria','Finland', 'Ireland', 
         'Portugal','Spain', 'Greece','Sweden','Malta','Luxembourg','Cyprus','Poland','Czech Republic','Romania','Hungary',
          'Slovakia','Croatia','Slovenia','Bulgaria', 'Latvia','Estonia','Lithuania']

In [ ]:
scimago_IBEROAMERICA = ['Spain', 'Brazil','Mexico','Portugal','Argentina', 'Chile', 'Colombia',
                'Cuba','Venezuela','Peru', 'Ecuador','Uruguay','Costa Rica','Panama',
                'Trinidad and Tobago','Jamaica','Bolivia','Guatemala','Paraguay',
                'Barbados','Dominican Republic','Honduras','Grenada','Nicaragua',
                'El Salvador','Haiti','Bermuda','St. Kitts and Nevis','Bahamas',
                'Guyana','Suriname','Belize','Dominica','Andorra','Antigua and Barbuda',
                'St. Vincent and the Grenadines','St. Lucia','Anguilla','Curacao']                
# Puerto Rico  # Guadeloupe  # French Guiana # Martinique # Falkland Islands (Malvinas)
# Netherlands Antilles  # Cayman Islands # Virgin Islands (U.S.) # Aruba   # Virgin Islands (British)
# Montserrat # Turks and Caicos Islands # South Georgia and the South Sandwich Islands
# Sint Maarten

In [ ]:
scimago_LANIC = ['Curacao','Cuba','Trinidad and Tobago','Jamaica','Barbados','Dominican Republic',
         'Grenada','Haiti','St. Kitts and Nevis','St. Vincent and the Grenadines',
         'Bahamas','Dominica','Antigua and Barbuda','St. Lucia']
# Puerto Rico  # Guadeloupe # Martinique  # Cayman Islands # Virgin Islands (U.S.)
# Aruba  # Virgin Islands (British)  # Sint Maarten

In [ ]:
scimago_NORDIC_COUNTRIES = ['Sweden','Denmark','Finland','Norway','Iceland']

In [ ]:
scimago_OECD = ['United States','United Kingdom','Germany','France', 'Italy','Belgium','Denmark','Netherlands','Austria','Finland',
        'Portugal','Spain', 'Ireland','Greece','Sweden','Latvia','Estonia','Slovenia','Lithuania','Poland','Czech Republic',
        'Luxembourg','Norway','Iceland','Japan','Canada','Australia','South Korea',
        'Switzerland','Turkey','Israel','Mexico','New Zealand','Hungary','Chile',
         'Colombia','Slovakia','Costa Rica']

In [ ]:
#https://datahelpdesk.worldbank.org/knowledgebase/articles/906519-world-bank-country-and-lending-groups
    
# BY REGION

In [ ]:
wb_east_asia_and_pacific = ['South Korea','Philippines','Australia', 'Laos', 'Samoa','Brunei Darussalam','Singapore', 'Cambodia',
                         'Malaysia','Solomon Islands','China','Marshall Islands','Taiwan','Fiji','Micronesia, Fed. Sts.',
                         'Thailand','Mongolia','Timor-Leste','Myanmar','Papua New Guinea','Tonga','Indonesia','New Caledonia',
                         'Japan','New Zealand','Vanuatu','Vietnam','North Korea']
#Kiribati  French Polynesia Guam Nauru Tuvalu Palau  Northern Mariana Islands

In [ ]:
wb_europe_and_central_asia = ['Albania','Norway','Andorra','Greece','Poland','Armenia','Portugal','Austria','Hungary','Romania',
                           'Azerbaijan','Iceland','Russia','Belarus','Ireland','San Marino','Belgium','Serbia',
                           'Bosnia and Herzegovina','Italy','Slovakia','Bulgaria','Kazakhstan','Slovenia','Kosovo','Spain',
                           'Croatia','Kyrgyz Republic','Sweden','Cyprus','Latvia','Switzerland','Czech Republic',
                           'Liechtenstein','Tajikistan','Denmark','Lithuania','Turkey','Estonia','Luxembourg','Turkmenistan',
                           'Moldova','Ukraine','Finland','Monaco','United Kingdom','France','Montenegro','Uzbekistan','Georgia',
                           'Netherlands','Germany','Macedonia']
# Gibraltar ,Greenland   Isle of Man  Channel Islands    Faroe Islands

In [ ]:
wb_latin_america_and_the_caribbean = ['Antigua and Barbuda','Curacao','Paraguay','Argentina','Dominica','Peru','Dominican Republic',
                                   'Bahamas','Ecuador','Barbados','El Salvador','St. Kitts and Nevis','Belize','Grenada',
                                   'St. Lucia', 'Bolivia','Guatemala','Brazil','Guyana','St. Vincent and the Grenadines','Haiti',
                                   'Suriname','Honduras','Trinidad and Tobago','Chile','Jamaica','Colombia','Mexico','Uruguay',
                                   'Costa Rica','Nicaragua','Venezuela','Cuba','Panama']
# Puerto Rico  	Sint Maarten (Dutch part) 	St. Martin (French part) 	Virgin Islands (U.S.) 
# British Virgin Islands  Cayman Islands  Turks and Caicos Islands

In [ ]:
wb_middle_east_and_north_africa = ['Algeria','Jordan','Qatar','Bahrain','Kuwait','Saudi Arabia','Djibouti','Lebanon','Syria','Egypt',
                                'Libya','Tunisia','Iran','Malta','United Arab Emirates','Iraq','Morocco','Israel','Oman','Yemen']
# 	West Bank and Gaza

In [ ]:
wb_middle_east_and_north_africa_africa = ['Algeria','Djibouti','Egypt','Libya','Tunisia','Morocco']
# 	West Bank and Gaza

In [ ]:
wb_middle_east_and_north_africa_others = ['Jordan','Qatar','Bahrain','Kuwait','Saudi Arabia','Lebanon','Syria','Iran','Malta',
                                       'United Arab Emirates','Iraq','Israel','Oman','Yemen']


In [ ]:
wb_north_america = ['Bermuda','Canada','United States']

In [ ]:
wb_south_asia = ['Afghanistan','India','Pakistan','Bangladesh','Maldives','Sri Lanka','Bhutan','Nepal']

In [ ]:
wb_sub_saharan_africa = ['Angola','Ethiopia','Niger','Benin','Gabon', 'Nigeria','Botswana','Gambia', 'Rwanda','Burkina Faso','Ghana',
                      'Sao Tome and Principe','Burundi','Guinea','Senegal','Cabo Verde', 'Guinea-Bissau','Seychelles','Cameroon',
                      'Kenya','Sierra Leone','Central African Republic','Lesotho','Somalia','Chad','Liberia','South Africa',
                      'Comoros','Madagascar','South Sudan','DR Congo','Malawi','Sudan','Congo Republic','Mali','Tanzania',
                      "Cote d'Ivoire",'Mauritania','Togo','Equatorial Guinea','Mauritius','Uganda','Eritrea','Mozambique','Zambia',
                      'Eswatini','Namibia','Zimbabwe']

In [ ]:
# BY INCOME

In [ ]:
wb_low_income_economies = ['Afghanistan','Guinea-Bissau','Somalia','Burkina Faso','North Korea','South Sudan','Burundi','Liberia',
                        'Sudan','Central African Republic','Madagascar','Syria','Chad','Malawi','Togo','DR Congo','Mali','Uganda',
                        'Eritrea','Mozambique','Yemen','Ethiopia','Niger','Zambia','Gambia', 'Rwanda','Guinea','Sierra Leone']

In [ ]:
wb_low_income_economies_africa = ['Guinea-Bissau','Somalia','Burkina Faso','South Sudan','Burundi','Liberia',
                        'Sudan','Central African Republic','Madagascar','Chad','Malawi','Togo','DR Congo','Mali','Uganda',
                        'Eritrea','Mozambique','Ethiopia','Niger','Zambia','Gambia', 'Rwanda','Guinea','Sierra Leone']

In [ ]:
wb_low_income_economies_others = ['Afghanistan','North Korea','Syria','Yemen']

In [ ]:
wb_lower_middle_income_economies = ['Angola', 'India', 'Philippines','Algeria', 'Indonesia','Samoa','Bangladesh', 'Iran',
                                 'Sao Tome and Principe','Benin','Kenya', 'Senegal','Bhutan','Solomon Islands','Bolivia',
                                 'Kyrgyz Republic','Sri Lanka','Cabo Verde', 'Laos','Tanzania', 'Cambodia','Lebanon','Tajikistan',
                                 'Cameroon','Lesotho','Timor-Leste','Comoros','Mauritania','Tunisia','Congo Republic',
                                 'Micronesia, Fed. Sts.','Ukraine',"Cote d'Ivoire",'Mongolia','Uzbekistan','Djibouti','Morocco',
                                 'Vanuatu','Egypt','Myanmar','Vietnam','El Salvador','Nepal','Eswatini','Nicaragua','Zimbabwe',
                                 'Ghana','Nigeria','Haiti','Pakistan','Honduras','Papua New Guinea']
#Kiribati 	West Bank and Gaza 

In [ ]:
wb_lower_middle_income_economies_africa = ['Angola','Algeria','Sao Tome and Principe','Benin','Kenya', 'Senegal','Cabo Verde', 
                                        'Tanzania','Cameroon','Lesotho','Comoros','Mauritania','Tunisia','Congo Republic',
                                        "Cote d'Ivoire",'Djibouti','Morocco','Egypt','Eswatini','Zimbabwe','Ghana','Nigeria']
# 	West Bank and Gaza 

In [ ]:
wb_lower_middle_income_economies_others = ['India','Philippines','Indonesia','Samoa','Bangladesh','Iran','Bhutan','Solomon Islands',
                                        'Bolivia','Kyrgyz Republic','Sri Lanka','Laos','Cambodia','Lebanon','Tajikistan',
                                        'Timor-Leste','Micronesia, Fed. Sts.','Ukraine','Mongolia','Uzbekistan','Vanuatu','Myanmar',
                                        'Vietnam','El Salvador','Nepal','Nicaragua','Haiti','Pakistan','Honduras','Papua New Guinea']
#Kiribati 	West Bank and Gaza 

In [ ]:
wb_upper_middle_income_economies = ['Albania','Fiji','Namibia','Gabon','Macedonia','Argentina','Georgia','Armenia','Grenada',
                                 'Paraguay','Azerbaijan','Guatemala','Peru','Belarus','Guyana','Russia','Belize','Iraq','Serbia',
                                 'Bosnia and Herzegovina','Jamaica','South Africa','Botswana','Jordan','St. Lucia','Brazil',
                                 'Kazakhstan','St. Vincent and the Grenadines','Bulgaria','Kosovo','Suriname','China','Libya',
                                 'Thailand','Colombia','Malaysia','Tonga','Costa Rica','Maldives','Turkey','Cuba','Marshall Islands',
                                 'Turkmenistan', 'Dominica','Mauritius','Dominican Republic','Mexico','Equatorial Guinea','Moldova',
                                 'Ecuador','Montenegro']
# 'Tuvalu' 	Palau 

In [ ]:
wb_upper_middle_income_economies_africa = ['Namibia','Gabon','South Africa','Botswana','Libya','Mauritius','Equatorial Guinea']

In [ ]:
wb_upper_middle_income_economies_others = ['Albania','Fiji','Macedonia','Argentina','Georgia','Armenia','Grenada','Paraguay',
                                        'Azerbaijan','Guatemala','Peru','Belarus','Guyana','Russia','Belize','Iraq','Serbia',
                                        'Bosnia and Herzegovina','Jamaica','Jordan','St. Lucia','Brazil','Kazakhstan',
                                        'St. Vincent and the Grenadines','Bulgaria','Kosovo','Suriname','China','Thailand',
                                        'Colombia','Malaysia','Tonga','Costa Rica','Maldives','Turkey','Cuba','Marshall Islands',
                                        'Turkmenistan', 'Dominica','Dominican Republic','Mexico','Moldova','Ecuador','Montenegro']
# 'Tuvalu' 	Palau 

In [ ]:
wb_high_income_economies = [ 'Andorra','Greece','Poland','Antigua and Barbuda', 'Portugal','Australia','Qatar','Austria','Hungary',
                         'Romania','Bahamas','Iceland','San Marino','Bahrain','Ireland','Saudi Arabia','Barbados','Seychelles',
                         'Belgium','Israel','Singapore','Bermuda','Italy','Japan','Slovakia','Brunei Darussalam','South Korea',
                         'Slovenia','Canada','Kuwait','Spain','Latvia','St. Kitts and Nevis','Liechtenstein','Chile','Lithuania',
                         'Sweden','Croatia','Luxembourg','Switzerland','Curacao','Taiwan','Cyprus','Malta','Trinidad and Tobago',
                         'Czech Republic','Monaco','Denmark','United Arab Emirates','Estonia','Netherlands','United Kingdom',
                         'New Caledonia','United States','Finland','New Zealand','Uruguay','France','Norway','Germany','Oman',
                         'Panama']
# Channel Islands 	Gibraltar Aruba 	Northern Mariana Islands 	Virgin Islands (U.S.)	Nauru 	Sint Maarten (Dutch part) 	Isle of Man 	British Virgin Islands 	
# Cayman Islands 	French Polynesia  Faroe Islands  	Turks and Caicos Islands 	St. Martin (French part)	'Guam' 	Puerto Rico, Greenland,

In [ ]:
wb_high_income_economies_africa = ['Seychelles']

In [ ]:
wb_high_income_economies_others = [ 'Andorra','Greece','Poland','Antigua and Barbuda', 'Portugal','Australia','Qatar','Austria','Hungary',
                         'Romania','Bahamas','Iceland','San Marino','Bahrain','Ireland','Saudi Arabia','Barbados',
                         'Belgium','Israel','Singapore','Bermuda','Italy','Japan','Slovakia','Brunei Darussalam','South Korea',
                         'Slovenia','Canada','Kuwait','Spain','Latvia','St. Kitts and Nevis','Liechtenstein','Chile','Lithuania',
                         'Sweden','Croatia','Luxembourg','Switzerland','Curacao','Taiwan','Cyprus','Malta','Trinidad and Tobago',
                         'Czech Republic','Monaco','Denmark','United Arab Emirates','Estonia','Netherlands','United Kingdom',
                         'New Caledonia','United States','Finland','New Zealand','Uruguay','France','Norway','Germany','Oman',
                         'Panama']
# Channel Islands 	Gibraltar Aruba 	Northern Mariana Islands 	Virgin Islands (U.S.)	Nauru 	Sint Maarten (Dutch part) 	Isle of Man 	British Virgin Islands 	
# Cayman Islands 	French Polynesia  Faroe Islands  	Turks and Caicos Islands 	St. Martin (French part)	'Guam' 	Puerto Rico, Greenland,

In [ ]:
# BY LENDING

In [ ]:
wb_IDA = ['Afghanistan','Haiti','Rwanda','Bangladesh','Honduras','Samoa','Benin','Sao Tome and Principe','Bhutan','Kosovo','Senegal',
       'Burkina Faso', 'Kyrgyz Republic','Sierra Leone','Burundi','Laos','Solomon Islands', 'Cambodia','Lesotho','Somalia', 
       'Central African Republic','Liberia','South Sudan','Chad','Madagascar','Sudan','Comoros','Malawi','Syria','DR Congo',
       'Maldives','Tajikistan',"Cote d'Ivoire",'Mali','Tanzania','Djibouti','Marshall Islands','Togo','Eritrea','Mauritania',
       'Tonga','Ethiopia','Micronesia, Fed. Sts.','Gambia','Mozambique','Uganda','Ghana','Myanmar','Vanuatu','Guinea',
       'Nepal','Yemen', 'Guinea-Bissau','Nicaragua','Zambia','Guyana','Niger']

In [ ]:
wb_IDA_africa = ['Rwanda','Benin','Sao Tome and Principe','Senegal','Burkina Faso','Sierra Leone','Burundi','Lesotho','Somalia',
              'Central African Republic','Liberia','South Sudan','Chad','Madagascar','Sudan','Comoros','Malawi','DR Congo',
              "Cote d'Ivoire",'Mali','Tanzania','Djibouti','Togo','Eritrea','Mauritania','Ethiopia','Gambia','Mozambique',
              'Uganda','Ghana','Guinea','Guinea-Bissau','Zambia','Niger']

In [ ]:
wb_IDA_others = ['Afghanistan','Haiti','Bangladesh','Honduras','Samoa','Bhutan','Kosovo','Kyrgyz Republic','Laos','Solomon Islands', 
              'Cambodia','Syria','Maldives','Tajikistan','Marshall Islands','Tonga','Micronesia, Fed. Sts.','Myanmar',
              'Vanuatu','Nepal','Yemen', 'Nicaragua','Guyana']

In [ ]:
wb_Blend = ['Cabo Verde','Grenada','St. Lucia','Cameroon','Kenya','St. Vincent and the Grenadines', 'Congo Republic', 
         'Nigeria','Timor-Leste', 'Dominica','Pakistan','Uzbekistan','Fiji','Papua New Guinea','Zimbabwe']

In [ ]:
wb_Blend_africa = ['Cabo Verde','Cameroon','Kenya','Congo Republic','Nigeria','Zimbabwe']

In [ ]:
wb_Blend_others = ['Grenada','St. Lucia','St. Vincent and the Grenadines','Timor-Leste', 'Dominica','Pakistan','Uzbekistan',
                'Fiji','Papua New Guinea']

In [ ]:
wb_IBRD = ['Albania','Eswatini','Panama','Algeria','Gabon','Paraguay','Angola','Georgia','Peru','Antigua and Barbuda','Guatemala',
        'Philippines','Argentina','India','Poland','Armenia','Indonesia','Romania','Azerbaijan','Iran','Russia','Belarus','Iraq',
        'Serbia','Belize','Jamaica' ,'Seychelles','Bolivia','Jordan','South Africa','Bosnia and Herzegovina','Kazakhstan',
        'Sri Lanka','Botswana','Lebanon','St. Kitts and Nevis','Brazil','Libya','Suriname','Bulgaria','Malaysia','Thailand',
        'Chile','Mauritius','Trinidad and Tobago','China','Mexico','Tunisia','Colombia','Moldova' ,'Turkey','Costa Rica',
        'Mongolia','Turkmenistan','Croatia','Montenegro','Ukraine','Dominican Republic','Morocco' ,'Uruguay','Ecuador','Namibia',
        'Venezuela', 'Egypt','Vietnam','El Salvador','Macedonia','Equatorial Guinea']
# Nauru  Palau

In [ ]:
wb_IBRD_africa = ['Algeria','Gabon','Angola','Seychelles','South Africa','Botswana','Libya','Mauritius','Tunisia','Morocco' ,
               'Namibia','Egypt','Equatorial Guinea']

In [ ]:
wb_IBRD_others = ['Albania','Eswatini','Panama','Paraguay','Georgia','Peru','Antigua and Barbuda','Guatemala',
        'Philippines','Argentina','India','Poland','Armenia','Indonesia','Romania','Azerbaijan','Iran','Russia','Belarus','Iraq',
        'Serbia','Belize','Jamaica' ,'Bolivia','Jordan','Bosnia and Herzegovina','Kazakhstan',
        'Sri Lanka','Lebanon','St. Kitts and Nevis','Brazil','Suriname','Bulgaria','Malaysia','Thailand',
        'Chile','Trinidad and Tobago','China','Mexico','Colombia','Moldova' ,'Turkey','Costa Rica',
        'Mongolia','Turkmenistan','Croatia','Montenegro','Ukraine','Dominican Republic','Uruguay','Ecuador',
        'Venezuela','Vietnam','El Salvador','Macedonia']
# Nauru  Palau

In [ ]:
# others classification

In [ ]:
# Langues

In [ ]:
lan_afrikaans = ['Namibia', 'South Africa']

In [ ]:
lan_amharique = ['Ethiopia']

In [ ]:
lan_anglais = ['South Africa', 'Namibia', 'Cameroon', 'Zimbabwe','Botswana','Gambia','Ghana','Kenya', 'Lesotho','Liberia',
               'Malawi','Mauritius','Nigeria','Uganda','Seychelles','Sierra Leone','South Sudan','Eswatini','Tanzania','Zambia']

In [ ]:
lan_arabe = ['Algeria','Djibouti','Egypt','Eritrea','Libya','Morocco', 'Mauritania', 'Sudan','Chad','Tunisia','Comoros']

In [ ]:
lan_espagnol = ['Equatorial Guinea']

In [ ]:
lan_français = ['Benin','Burkina Faso', 'Gabon','Burundi','Cameroon','Central African Republic','Comoros','Congo Republic',
            'DR Congo', "Cote d'Ivoire",'Mali','Niger','Rwanda', 'Guinea', 'Senegal','Togo', 'Madagascar', 'Equatorial Guinea',
            'Chad', 'Mauritania', 'Seychelles']

In [ ]:
lan_malgache = ['Madagascar']

In [ ]:
lan_portugais = ['Angola','Cabo Verde', 'Mozambique','Sao Tome and Principe','Guinea-Bissau']

In [ ]:
lan_somali = ['Somalia']

In [ ]:
lan_kinyarwanda = ['Rwanda']

In [ ]:
lan_kirundi = ['Burundi']

In [ ]:
lan_tigrinia = ['Eritrea']

In [ ]:
lan_swahili = ['Kenya', 'Tanzania']

In [ ]:
lan_swati = ['Eswatini']

In [ ]:
lan_creole = ['Seychelles']

In [ ]:
lan_sesotho = ['Lesotho']

In [ ]:
# histoire coloniale

In [ ]:
col_Germany = ['Namibia']

In [ ]:
col_Belgique = ['Burundi', 'DR Congo', 'Rwanda']

In [ ]:
col_Espagne = ['Equatorial Guinea']

In [ ]:
col_France = ['Cameroon','Algeria','Benin','Burkina Faso', 'Gabon','Central African Republic','Chad','Comoros','Congo Republic',
          "Cote d'Ivoire",'Djibouti','Guinea', 'Madagascar', 'Mauritania', 'Morocco', 'Mali', 'Niger', 'Senegal', 'Togo', 
          'Tunisia']

In [ ]:
col_United_Kingdom = ['Zimbabwe','Eritrea', 'Somalia', 'Cameroon', 'Sudan', 'Botswana', 'Egypt','Gambia','Ghana','Kenya', 'Lesotho',
                  'Malawi', 'Nigeria', 'Seychelles', 'Sierra Leone', 'South Africa', 'Eswatini','Tanzania', 'Uganda', 'Zambia']

In [ ]:
col_Italie = ['Eritrea', 'Libya', 'Somalia']

In [ ]:
col_Portugal = ['Angola', 'Cabo Verde', 'Mozambique', 'Sao Tome and Principe']


### count group

In [ ]:
groups_values = [north_africa, eastern_africa, Southern_africa, Western_africa, Central_africa, non_african_countries, 
                 scimago_asiatic_region, scimago_eastern_europe, scimago_latin_america,scimago_middle_east, scimago_northern_america, scimago_pacific_region, scimago_western_europe, scimago_arab_countries, scimago_arab_countries_non_africa,scimago_arab_countries_africa, scimago_BRIICS, scimago_BRIICS_not_south_africa, scimago_EU_28,scimago_IBEROAMERICA, scimago_LANIC, scimago_NORDIC_COUNTRIES, scimago_OECD, 
                 wb_east_asia_and_pacific, wb_europe_and_central_asia, wb_latin_america_and_the_caribbean, wb_middle_east_and_north_africa, wb_middle_east_and_north_africa_africa, wb_middle_east_and_north_africa_others, wb_north_america, wb_south_asia, wb_sub_saharan_africa,
                 wb_low_income_economies, wb_low_income_economies_africa, wb_low_income_economies_others, wb_lower_middle_income_economies, wb_lower_middle_income_economies_africa, wb_lower_middle_income_economies_others, wb_upper_middle_income_economies, wb_upper_middle_income_economies_africa, wb_upper_middle_income_economies_others, wb_high_income_economies, wb_high_income_economies_africa, wb_high_income_economies_others,
                 wb_IDA, wb_IDA_africa, wb_IDA_others, wb_Blend, wb_Blend_africa, wb_Blend_others, wb_IBRD, wb_IBRD_africa, wb_IBRD_others, 
                 lan_afrikaans, lan_amharique, lan_anglais, lan_arabe, lan_espagnol, lan_français, lan_malgache, lan_portugais, lan_somali, lan_kinyarwanda, lan_kirundi, lan_tigrinia, lan_swahili, lan_swati, lan_creole, lan_sesotho, 
                 col_Germany, col_Belgique, col_Espagne, col_France, col_United_Kingdom, col_Italie, col_Portugal]

In [ ]:
groups_values

In [ ]:
groups_keys = ['north_africa', 'eastern_africa', 'Southern_africa', 'Western_africa', 'Central_africa', 'non_african_countries', 
                 'scimago_asiatic_region', 'scimago_eastern_europe', 'scimago_latin_america','scimago_middle_east', 'scimago_northern_america', 'scimago_pacific_region', 'scimago_western_europe', 'scimago_arab_countries', 'scimago_arab_countries_non_africa','scimago_arab_countries_africa', 'scimago_BRIICS', 'scimago_BRIICS_not_south_africa', 'scimago_EU_28','scimago_IBEROAMERICA', 'scimago_LANIC', 'scimago_NORDIC_COUNTRIES', 'scimago_OECD', 
                 'wb_east_asia_and_pacific', 'wb_europe_and_central_asia', 'wb_latin_america_and_the_caribbean', 'wb_middle_east_and_north_africa', 'wb_middle_east_and_north_africa_africa', 'wb_middle_east_and_north_africa_others', 'wb_north_america', 'wb_south_asia', 'wb_sub_saharan_africa',
                 'wb_low_income_economies', 'wb_low_income_economies_africa', 'wb_low_income_economies_others', 'wb_lower_middle_income_economies', 'wb_lower_middle_income_economies_africa', 'wb_lower_middle_income_economies_others', 'wb_upper_middle_income_economies', 'wb_upper_middle_income_economies_africa', 'wb_upper_middle_income_economies_others', 'wb_high_income_economies', 'wb_high_income_economies_africa', 'wb_high_income_economies_others',
                 'wb_IDA', 'wb_IDA_africa', 'wb_IDA_others', 'wb_Blend', 'wb_Blend_africa', 'wb_Blend_others', 'wb_IBRD', 'wb_IBRD_africa', 'wb_IBRD_others', 
                 'lan_afrikaans', 'lan_amharique', 'lan_anglais', 'lan_arabe', 'lan_espagnol', 'lan_français', 'lan_malgache', 'lan_portugais', 'lan_somali', 'lan_kinyarwanda', 'lan_kirundi', 'lan_tigrinia', 'lan_swahili', 'lan_swati', 'lan_creole', 'lan_sesotho', 
                 'col_Germany', 'col_Belgique', 'col_Espagne', 'col_France', 'col_United_Kingdom', 'col_Italie', 'col_Portugal']

In [ ]:
groups_keys

In [ ]:
# using zip()
# to convert lists to dictionary
groups_keys_values = dict(zip(groups_keys, groups_values))

In [ ]:
groups_keys_values

In [ ]:
groups_keys_values['north_africa']

In [ ]:
for group_key in groups_keys:
    data_concat[group_key] = data_concat[groups_keys_values[group_key]].count(axis=1)

In [ ]:
data_concat.shape

In [ ]:
data.shape

In [ ]:
data_concat = data_concat.replace({0: np.nan})

In [ ]:
#save data
data_concat.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/studies_with_african_all_based_on_affil_only_and_group.csv')
data_concat

# import data

In [ ]:
data_concat = pd.read_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/studies_with_african_all_based_on_affil_only_and_group.csv')
data_concat

In [ ]:
data_prep_all = pd.read_csv('D:\RESEARCH IN AFRICA\Manuscripts/comprarison of databases/data/preprocessing_wos/concat_and_preprocessing_wos.csv')
data_prep_all = data_prep_all.drop_duplicates('UT (Unique WOS ID)')
data_prep_all

## quels sont les pays ayant contribués et leurs contributions

In [ ]:
contries_df = data_concat.count().sort_values(ascending=False).to_frame().rename(columns={0:'count'})
contries_df

In [ ]:
contries_df.head(60)

In [ ]:

contries_df['percentage'] = contries_df['count']/1271941*100
contries_df

In [ ]:
contries_df.head(60)

In [ ]:
#save data
contries_df.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/contries_participation.csv')


# evolution of research in Africa

1. Évolution de la recherche en Afrique de 1975 à 2021

Évolution de  la quantité de recherche pour :
- Global
- ou par région
- Par pays 

In [ ]:
year_count = data_concat['year'].fillna('NA').value_counts().to_frame().reset_index().rename(columns={'index':'year','year':'count'})
#save data
#year_count.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/year_count.csv')

year_count

In [ ]:
year_count_dropna= year_count[year_count.year != 'NA']

In [ ]:
from sklearn.linear_model import Ridge

lr = Ridge()
lr.fit(year_count_dropna[['year']], year_count_dropna['count'])

# Ridge(alpha=1.0, copy_X=True, fit_intercept=True, max_iter=None,
#    normalize=False, random_state=None, solver='auto', tol=0.001)

plt.bar(year_count_dropna['year'], year_count_dropna['count'])

plt.plot(year_count_dropna['year'], lr.coef_*year_count_dropna['year']+lr.intercept_, color='orange')

## Build bar plot graph
plt.title("Research trend in Africa")
plt.xlabel("Publication years")
plt.ylabel("Number of publication")

# Save files in pdf and eps format
# plt.savefig("../paper/figures/Fig.1 Research trend in Africa.jpeg", dpi=300)
# plt.savefig("../paper/figures/Fig.1 Research trend in Africa.png", dpi=300)

In [ ]:
data_concat_year = data_concat.drop(['PubMed_ID_wos','doi','source_title'], axis=1).set_index('wos_ID')

data_concat_year

In [ ]:
data_concat_year_table = data_concat_year.groupby(['year']).count()
#save data
#year_count.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/year_count_countries.csv')

data_concat_year_table

In [ ]:
data_concat_year_group_table = data_concat_year.groupby(['year_group']).count()

#save data
#year_count.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/year_group_count_countries.csv')

data_concat_year_group_table

# Types de documents

Ici il faudrait définir et comparer l'évolution de la proportion des types de documents par région :
- par année et par pays
- ou par région
- ou global 

In [ ]:
# document contries

In [ ]:
data_prep_all_doc_types = data_prep_all[['UT (Unique WOS ID)','Document Type']].rename(columns={'UT (Unique WOS ID)':'wos_ID'})
data_prep_all_doc_types['wos_ID'] = data_prep_all_doc_types['wos_ID'].str.lower()

In [ ]:
data_concat_doc_types = data_concat.merge(data_prep_all_doc_types, on = 'wos_ID', how='left')
data_concat_doc_types

In [ ]:
data_concat_doc_types_table = data_concat_doc_types.groupby(['Document Type']).count()

#save data
#data_concat_doc_types_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/doc_types_table_count_countries.csv')

data_concat_doc_types_table

In [ ]:
# document contries

In [ ]:
data_concat_doc_types2 = data_concat_doc_types[['year','year_group','Document Type']].set_index(['year_group','year'])
data_concat_doc_types2

In [ ]:
# get dummies to have data on columns
dfs = pd.get_dummies(data_concat_doc_types2['Document Type']).reset_index()
#save data
#dfs.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/doc_types_table_count_years.csv')

dfs

In [ ]:
data_concat_doc_types_year_table = dfs.groupby(['year']).count()
#save data
#data_concat_doc_types_year_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/doc_types_table_count_years.csv')

data_concat_doc_types_year_table

In [ ]:
data_concat_doc_types_year_g_table = dfs.groupby(['year_group']).count()
#save data
#data_concat_doc_types_year_g_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/doc_types_table_count_years_group.csv')

data_concat_doc_types_year_g_table

# Publisher, Journal and  caracteristics


Quels sont les caractéristiques des journaux dans lesquels les pays publient et comment ça évoluent?:
- impact factor
- local, régional, continental ou autre,
- open Access journal ou non
- publisher and cities
- APC free, or not, solution for African countries or not 

# Langues

Proposition and evolution of publications language according to countries, région, Global 

# Funding

Proposition and evolution of publications receiving funding according to région, countries and global
Top 5 per région 

# Thématiques

Wich thème are most publish on, according to countries, région and global,
And how they evolution 

# Citations

Which region or countries are more attractive and received more Impact


# Collaboration scientifique

Proportion of local, regional, continental and international collaboration and evolution during years



# Nombre des auteurs
Évolution of authors numbers per studies, according to region

# Auteurs Africain en premier et dernier position et auteurs correspondant
Évolution of the proportion of African authors at the leading position according to the region:
- first
- last
- correspondant